
# 51job 职位信息抓取 Notebook

功能：

1. 读取 CSV 文件
2. 遍历 “标题链接” 列
3. 打开每个 51job 职位详情页
4. 抓取 “职位信息” 模块
5. 保存到新的 CSV 文件


In [ ]:

!pip install pandas playwright nest_asyncio


In [ ]:

!playwright install


In [ ]:

import pandas as pd
import asyncio
import nest_asyncio
import random
from playwright.async_api import async_playwright

nest_asyncio.apply()



## 修改这里的 CSV 文件名

你的 CSV 至少需要包含：

- 标题
- 标题链接


In [ ]:

csv_path = "jobs.csv"   # 修改成你的文件名

df = pd.read_csv(csv_path)

print(df.head())

# 新增列
df["职位信息"] = ""



## 开始抓取职位信息


In [ ]:

async def scrape_job_info():

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=False,
            slow_mo=300
        )

        page = await browser.new_page()

        for i, row in df.iterrows():

            url = row["标题链接"]

            try:

                print(f"\n正在抓取: {url}")

                await page.goto(
                    url,
                    timeout=60000
                )

                # 随机等待
                await page.wait_for_timeout(
                    random.randint(3000, 6000)
                )

                # =========================
                # 抓取职位信息模块
                # =========================

                job_info = ""

                selectors = [
                    ".jobdesc",
                    ".jobdetail-box",
                    ".desc",
                    ".job_msg",
                    ".bmsg",
                    ".job-detail"
                ]

                found = False

                for selector in selectors:

                    try:

                        element = await page.query_selector(selector)

                        if element:

                            text = await element.inner_text()

                            if len(text.strip()) > 30:

                                job_info = text.strip()

                                found = True

                                print("成功抓取职位信息")

                                break

                    except:
                        pass

                if not found:

                    print("未找到职位信息模块")

                    job_info = ""

                # 保存
                df.at[i, "职位信息"] = job_info

            except Exception as e:

                print(f"错误: {e}")

                df.at[i, "职位信息"] = ""

        await browser.close()



## 运行爬虫


In [ ]:

asyncio.run(scrape_job_info())



## 保存结果


In [ ]:

output_path = "jobs_with_jobinfo.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n已保存: {output_path}")



## 输出结果示例

新增字段：

- 职位信息
